# VENOZK STUDIO — Colab (T4)
Run **Cell 1** once, then **Cell 2** whenever you want to process files. Set everything in Cell 2's form.
Runtime → Change runtime type → **T4 GPU**.

In [ ]:
#@title **Cell 1 - Setup** { display-mode: "form" }
import os, sys, subprocess
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True)
gpu = sh("nvidia-smi -L").stdout.strip()
print(gpu or "WARNING: no GPU. Runtime > Change runtime type > T4 GPU")
print("[1/3] Installing packages...")
r = sh(f"{sys.executable} -m pip install -q spandrel 'rembg[gpu]' onnxruntime-gpu opencv-python-headless tqdm")
if r.returncode: print(r.stderr[-400:])
print("[2/3] Downloading open-source models (Real-ESRGAN, BSD-3)...")
os.makedirs("/content/models", exist_ok=True)
B = "https://github.com/xinntao/Real-ESRGAN/releases/download/"
MODELS = {
 "realesr-general-x4v3.pth":       B+"v0.2.5.0/realesr-general-x4v3.pth",
 "realesr-general-wdn-x4v3.pth":   B+"v0.2.5.0/realesr-general-wdn-x4v3.pth",
 "realesr-animevideov3.pth":       B+"v0.2.5.0/realesr-animevideov3.pth",
 "RealESRGAN_x2plus.pth":          B+"v0.2.1/RealESRGAN_x2plus.pth",
 "RealESRGAN_x4plus.pth":          B+"v0.1.0/RealESRGAN_x4plus.pth",
 "RealESRGAN_x4plus_anime_6B.pth": B+"v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth",
}
for f, u in MODELS.items():
    p = f"/content/models/{f}"
    if not (os.path.exists(p) and os.path.getsize(p) > 1_000_000):
        sh(f"wget -q -O {p} {u}")
    ok = os.path.exists(p) and os.path.getsize(p) > 1_000_000
    print(("  OK  " if ok else "  FAIL"), f, f"{os.path.getsize(p)//1048576} MB" if os.path.exists(p) else "")
print("[3/3] Checking ffmpeg...")
HAS_NVENC = int(sh("ffmpeg -hide_banner -encoders 2>/dev/null | grep -c nvenc").stdout.strip() or 0) > 0
print("  NVENC GPU encoding:", "available" if HAS_NVENC else "not available (CPU encoders will be used)")
print("[OK] Cell 1 complete. Now run Cell 2.")

In [ ]:
#@title **Cell 2 - Venozk Studio** { display-mode: "form" }
#@markdown ## 📁 Input / Output
MOUNT_DRIVE = False #@param {type:"boolean"}
INPUT_PATH = "" #@param {type:"string"}
#@markdown File or folder path (images and videos, batch works). Leave empty to upload from your device.
OUTPUT_DIR = "/content/output" #@param {type:"string"}
AUTO_DOWNLOAD = True #@param {type:"boolean"}
PREVIEW_ONLY = False #@param {type:"boolean"}
#@markdown Preview = process only the first frame/image and show before/after (test settings fast).
#@markdown ## 🧠 Upscale
UPSCALE = True #@param {type:"boolean"}
MODEL = "general-x4v3" #@param ["general-x4v3","x2plus","x4plus","anime-video-v3","anime-6B"]
#@markdown general-x4v3 = best all-round + denoise | x2plus/x4plus = clean photos | anime-* = cartoons
TARGET_SCALE = 2 #@param {type:"slider", min:1, max:4, step:0.5}
DENOISE = 0.5 #@param {type:"slider", min:0, max:1, step:0.05}
SHARPEN = 0.3 #@param {type:"slider", min:0, max:1.5, step:0.05}
TILE = 512 #@param [256,384,512,768,1024]
FP16 = True #@param {type:"boolean"}
#@markdown ## ✂️ Background removal
BG_MODEL = "none" #@param ["none","birefnet-general","birefnet-general-lite","birefnet-portrait","isnet-general-use","u2net_human_seg"]
BG_MODE = "transparent" #@param ["transparent","green","black","white","custom"]
BG_CUSTOM = "#0a1a3a" #@param {type:"string"}
#@markdown ## 🖼️ Image output
IMG_FORMAT = "png" #@param ["png","jpg","webp"]
IMG_QUALITY = 95 #@param {type:"slider", min:50, max:100, step:1}
#@markdown ## 🎬 Video output
VCODEC = "h264 (CPU)" #@param ["h264 (CPU)","h265 (CPU, smaller)","h264 NVENC (GPU, fast)","h265 NVENC (GPU, fast)","ProRes 422 HQ (editing)"]
CRF = 16 #@param {type:"slider", min:0, max:30, step:1}
PRESET = "slow" #@param ["ultrafast","fast","medium","slow","veryslow"]
OUTPUT_FPS = 0 #@param {type:"number"}
KEEP_AUDIO = True #@param {type:"boolean"}
START_SEC = 0 #@param {type:"number"}
DURATION_SEC = 0 #@param {type:"number"}
#@markdown FPS 0 = same as source. Duration 0 = whole video. Transparent BG on video exports VP9 WebM.

TILE=int(TILE); CRF=int(CRF); IMG_QUALITY=int(IMG_QUALITY); TARGET_SCALE=float(TARGET_SCALE); DENOISE=float(DENOISE); SHARPEN=float(SHARPEN); OUTPUT_FPS=float(OUTPUT_FPS); START_SEC=float(START_SEC); DURATION_SEC=float(DURATION_SEC)  # Colab dropdowns can return text
import os, glob, time, subprocess, cv2, numpy as np, torch
from tqdm.auto import tqdm
from spandrel import ModelLoader
from google.colab import files
from google.colab.patches import cv2_imshow
dev, dt = "cuda", (torch.float16 if FP16 else torch.float32)
FILES = {"general-x4v3":"realesr-general-x4v3","x2plus":"RealESRGAN_x2plus","x4plus":"RealESRGAN_x4plus","anime-video-v3":"realesr-animevideov3","anime-6B":"RealESRGAN_x4plus_anime_6B"}
def load(n):
    m = ModelLoader().load_from_file(f"/content/models/{n}.pth").to(dev).eval()
    return m.half() if FP16 else m
nets = []
if UPSCALE:
    nets = [load(FILES[MODEL])]
    if MODEL == "general-x4v3" and DENOISE > 0: nets.append(load("realesr-general-wdn-x4v3"))
even = lambda v: max(2, int(round(v/2))*2)

@torch.inference_mode()
def run_net(net, rgb, pad=16):
    s, (h, w) = net.scale, rgb.shape[:2]
    out = np.zeros((h*s, w*s, 3), np.uint8)
    for y in range(0, h, TILE):
        for x in range(0, w, TILE):
            y0, x0 = max(y-pad, 0), max(x-pad, 0)
            y1, x1 = min(y+TILE+pad, h), min(x+TILE+pad, w)
            t = torch.from_numpy(rgb[y0:y1, x0:x1]).to(dev).permute(2,0,1)[None].to(dt)/255
            o = net(t).clamp(0,1)[0].permute(1,2,0).float().mul(255).round().byte().cpu().numpy()
            th, tw, oy, ox = min(TILE,h-y)*s, min(TILE,w-x)*s, (y-y0)*s, (x-x0)*s
            out[y*s:y*s+th, x*s:x*s+tw] = o[oy:oy+th, ox:ox+tw]
    return out

def upscale(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    o = run_net(nets[0], rgb)
    if len(nets) > 1: o = cv2.addWeighted(o, 1-DENOISE, run_net(nets[1], rgb), DENOISE, 0)
    o = cv2.cvtColor(o, cv2.COLOR_RGB2BGR)
    h, w = bgr.shape[:2]; W, H = even(w*TARGET_SCALE), even(h*TARGET_SCALE)
    if (o.shape[1], o.shape[0]) != (W, H):
        o = cv2.resize(o, (W, H), interpolation=cv2.INTER_AREA if W < o.shape[1] else cv2.INTER_LANCZOS4)
    return o

sess = None
def mask_of(bgr):
    global sess
    from rembg import remove, new_session
    if sess is None: sess = new_session(BG_MODEL)
    m = np.array(remove(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), session=sess, only_mask=True))
    return m[..., 0] if m.ndim == 3 else m
hex2bgr = lambda h: tuple(int(h.lstrip("#")[i:i+2], 16) for i in (4, 2, 0))
COL = {"green":(0,255,0), "black":(0,0,0), "white":(255,255,255)}
ALPHA = BG_MODEL != "none" and BG_MODE == "transparent"

def process(bgr):
    m = mask_of(bgr) if BG_MODEL != "none" else None
    o = upscale(bgr) if UPSCALE else bgr
    if SHARPEN > 0: o = cv2.addWeighted(o, 1+SHARPEN, cv2.GaussianBlur(o, (0,0), 1.0), -SHARPEN, 0)
    if m is not None:
        m = cv2.resize(m, (o.shape[1], o.shape[0]), interpolation=cv2.INTER_CUBIC)
        if ALPHA: return np.dstack([o, m])
        c = np.array(hex2bgr(BG_CUSTOM) if BG_MODE == "custom" else COL[BG_MODE], np.float32)
        a = (m.astype(np.float32)/255)[..., None]
        o = (o*a + c*(1-a)).astype(np.uint8)
    return o

def show_cmp(a, b):
    b3 = b[..., :3]; a = cv2.resize(a, (b3.shape[1], b3.shape[0]), interpolation=cv2.INTER_NEAREST)
    v = np.hstack([a, b3]); k = min(1, 1600/v.shape[1])
    cv2_imshow(cv2.resize(v, None, fx=k, fy=k, interpolation=cv2.INTER_AREA))

def do_image(p, stem):
    bgr = cv2.imread(p, cv2.IMREAD_COLOR); o = process(bgr)
    if PREVIEW_ONLY: return show_cmp(bgr, o)
    fmt = "png" if (ALPHA and IMG_FORMAT == "jpg") else IMG_FORMAT
    q = {"png":[cv2.IMWRITE_PNG_COMPRESSION,3], "jpg":[cv2.IMWRITE_JPEG_QUALITY,IMG_QUALITY], "webp":[cv2.IMWRITE_WEBP_QUALITY,IMG_QUALITY]}[fmt]
    out = f"{OUTPUT_DIR}/{stem}_venozk.{fmt}"; cv2.imwrite(out, o, q); return out

def do_video(p, stem):
    cap = cv2.VideoCapture(p); fps = cap.get(cv2.CAP_PROP_FPS) or 30
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); s0 = int(START_SEC*fps)
    cap.set(cv2.CAP_PROP_POS_FRAMES, s0); n = min(n-s0, int(DURATION_SEC*fps)) if DURATION_SEC > 0 else n-s0
    ok, f = cap.read()
    if not ok: raise RuntimeError("Cannot read video")
    o = process(f)
    if PREVIEW_ONLY: return show_cmp(f, o)
    H, W = o.shape[:2]
    V = {"h264 (CPU)":("libx264",["-crf",str(CRF),"-preset",PRESET,"-pix_fmt","yuv420p"],"mp4"),
         "h265 (CPU, smaller)":("libx265",["-crf",str(CRF),"-preset",PRESET,"-pix_fmt","yuv420p","-tag:v","hvc1"],"mp4"),
         "h264 NVENC (GPU, fast)":("h264_nvenc",["-cq",str(CRF),"-preset","p5","-pix_fmt","yuv420p"],"mp4"),
         "h265 NVENC (GPU, fast)":("hevc_nvenc",["-cq",str(CRF),"-preset","p5","-pix_fmt","yuv420p","-tag:v","hvc1"],"mp4"),
         "ProRes 422 HQ (editing)":("prores_ks",["-profile:v","3","-pix_fmt","yuv422p10le"],"mov")}
    vc, va, ext = V[VCODEC]
    if "nvenc" in vc and not HAS_NVENC: vc, va, ext = V["h264 (CPU)"]; print("  NVENC unavailable, using h264 CPU")
    if ALPHA: vc, va, ext = "libvpx-vp9", ["-pix_fmt","yuva420p","-b:v","0","-crf",str(max(CRF,20)),"-row-mt","1"], "webm"
    ac = "libopus" if ext == "webm" else "aac"
    out = f"{OUTPUT_DIR}/{stem}_venozk.{ext}"
    cmd = ["ffmpeg","-y","-loglevel","error","-f","rawvideo","-pix_fmt","bgra" if ALPHA else "bgr24","-s",f"{W}x{H}","-r",str(fps),"-i","-"]
    if KEEP_AUDIO: cmd += ["-ss",str(START_SEC)] + (["-t",str(DURATION_SEC)] if DURATION_SEC > 0 else []) + ["-i",p]
    cmd += ["-map","0:v"] + (["-map","1:a?","-c:a",ac] if KEEP_AUDIO else []) + ["-c:v",vc,*va]
    if OUTPUT_FPS > 0: cmd += ["-r",str(OUTPUT_FPS)]
    cmd += ["-shortest", out]
    pr = subprocess.Popen(cmd, stdin=subprocess.PIPE)
    pr.stdin.write(np.ascontiguousarray(o).tobytes())
    for i in tqdm(range(n-1), desc=stem):
        ok, f = cap.read()
        if not ok: break
        pr.stdin.write(np.ascontiguousarray(process(f)).tobytes())
    pr.stdin.close(); pr.wait(); return out

# ---- collect inputs ----
if MOUNT_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
IMG_X, VID_X = {".png",".jpg",".jpeg",".webp",".bmp",".tif",".tiff"}, {".mp4",".mov",".mkv",".avi",".webm",".m4v"}
if not INPUT_PATH.strip(): paths = [os.path.abspath(n) for n in files.upload()]
elif os.path.isdir(INPUT_PATH): paths = sorted(glob.glob(INPUT_PATH.rstrip("/") + "/*"))
else: paths = [INPUT_PATH]
os.makedirs(OUTPUT_DIR, exist_ok=True)
if not UPSCALE and BG_MODEL == "none" and SHARPEN == 0: print("Nothing to do: enable UPSCALE, a BG model, or SHARPEN.")
results = []
t0 = time.time()
for p in paths:
    stem, ext = os.path.splitext(os.path.basename(p)); ext = ext.lower()
    try:
        if ext in IMG_X: r = do_image(p, stem)
        elif ext in VID_X: r = do_video(p, stem)
        else: print("Skipped (unsupported):", p); continue
        if r: results.append(r); print("Saved:", r)
    except Exception as e: print("FAILED", p, "->", e)
    torch.cuda.empty_cache()
print(f"Done in {time.time()-t0:.0f}s. Output folder: {OUTPUT_DIR}")
if AUTO_DOWNLOAD and not PREVIEW_ONLY:
    for r in results: files.download(r)